# Hosting LangGraph Agents with Groq models in Amazon Bedrock AgentCore Runtime
# 在 Amazon Bedrock AgentCore Runtime 中托管使用 Groq 模型的 LangGraph 智能体

## Overview
## 概述

In this tutorial we will learn how to host a FAQ inquiry agent using Amazon Bedrock AgentCore Runtime with Groq models.

在本教程中，我们将学习如何使用 Amazon Bedrock AgentCore Runtime 和 Groq 模型托管一个 FAQ 问询智能体。

This example demonstrates a RAG-based FAQ assistant for "Lauki Phones" that can search a knowledge base to answer customer questions.

本示例演示了一个基于 RAG 的「Lauki Phones」FAQ 助手，可以搜索知识库来回答客户问题。

### Tutorial Details
### 教程详情

| Information         | Details                                                                  |
|:--------------------|:-------------------------------------------------------------------------|
| Tutorial type       | Conversational FAQ                                                       |
| Agent type          | Single                                                                   |
| Agentic Framework   | LangGraph + LangChain                                                    |
| LLM model           | Groq Llama 3.3 70B                                                       |
| Tutorial components | RAG with FAISS, HuggingFace Embeddings, AgentCore Runtime                |
| Tutorial vertical   | Customer Service                                                         |
| Example complexity  | Medium                                                                   |
| SDK used            | Amazon BedrockAgentCore Python SDK, LangChain, LangGraph                 |

| 信息项              | 详情                                                                     |
|:--------------------|:-------------------------------------------------------------------------|
| 教程类型            | 对话式 FAQ                                                               |
| 智能体类型          | 单一智能体                                                               |
| 智能体框架          | LangGraph + LangChain                                                    |
| LLM 模型            | Groq Llama 3.3 70B                                                       |
| 教程组件            | 使用 FAISS 的 RAG、HuggingFace 嵌入、AgentCore Runtime                   |
| 教程适用领域        | 客户服务                                                                 |
| 示例复杂度          | 中等                                                                     |
| 使用的 SDK          | Amazon BedrockAgentCore Python SDK、LangChain、LangGraph                 |

## Prerequisites
## 前提条件

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* **Groq API Key** (Get it from https://console.groq.com/keys)
* Amazon Bedrock AgentCore SDK
* LangGraph & LangChain
* Docker running

要执行本教程，您需要：
* Python 3.10+
* AWS 凭证
* **Groq API 密钥**（从 https://console.groq.com/keys 获取）
* Amazon Bedrock AgentCore SDK
* LangGraph 和 LangChain
* Docker 运行中

In [20]:
%pip install -U -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Environment Setup
## 环境配置

Set your Groq API key. You can get one from https://console.groq.com/keys

设置您的 Groq API 密钥。您可以从 https://console.groq.com/keys 获取

In [19]:
import os
from dotenv import load_dotenv

# Load from .env file if exists
load_dotenv()

# Or set directly (replace with your actual key)
# os.environ["GROQ_API_KEY"] = "<YOUR_GROQ_API_KEY>"

# Verify the key is set
if not os.getenv("GROQ_API_KEY"):
    print("Warning: GROQ_API_KEY is not set!")
else:
    print("GROQ_API_KEY is configured.")

GROQ_API_KEY is configured.


## Creating your agents and experimenting locally
## 创建智能体并在本地进行实验

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

在将智能体部署到 AgentCore Runtime 之前，让我们先在本地开发和运行它们进行实验。

Our FAQ agent uses:
- **FAISS** for vector similarity search
- **HuggingFace Embeddings** for text embedding
- **ChatGroq** for fast LLM inference with Groq
- **LangGraph** for reliable tool-calling agent

我们的 FAQ 智能体使用：
- **FAISS** 进行向量相似度搜索
- **HuggingFace Embeddings** 进行文本嵌入
- **ChatGroq** 通过 Groq 实现快速 LLM 推理
- **LangGraph** 实现可靠的工具调用智能体

In [24]:
%%writefile langgraph_agents_groq.py
"""
FAQ Agent using LangChain + ChatGroq
This version uses LangGraph for reliable tool calling with Groq models.
"""
import csv
import os
import argparse
import json
from typing import List

from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv

# Load environment variables
_ = load_dotenv()


def load_faq_csv(path: str) -> List[Document]:
    """Load FAQ data from CSV file"""
    docs = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            q = row["question"].strip()
            a = row["answer"].strip()
            docs.append(Document(page_content=f"Q: {q}\nA: {a}"))
    return docs


# Initialize FAQ knowledge base
script_dir = os.path.dirname(os.path.abspath(__file__))
csv_path = os.path.join(script_dir, "lauki_qna.csv")
docs = load_faq_csv(csv_path)

# Setup embeddings and vector store
emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
chunks = splitter.split_documents(docs)
store = FAISS.from_documents(chunks, emb)


# Define tools for the agent
@tool
def search_faq(query: str) -> str:
    """Search the FAQ knowledge base for relevant information.
    Use this tool when the user asks questions about Lauki Phones products, services, or policies.

    Args:
        query: The search query to find relevant FAQ entries

    Returns:
        Relevant FAQ entries that might answer the question
    """
    results = store.similarity_search(query, k=3)

    if not results:
        return "No relevant FAQ entries found."

    context = "\n\n---\n\n".join([
        f"FAQ Entry {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(results)
    ])

    return f"Found {len(results)} relevant FAQ entries:\n\n{context}"


@tool
def search_detailed_faq(query: str) -> str:
    """Search the FAQ knowledge base with more results for complex queries.
    Use this when the initial search doesn't provide enough information.

    Args:
        query: The search query

    Returns:
        More comprehensive FAQ entries (5 results)
    """
    results = store.similarity_search(query, k=5)

    if not results:
        return "No relevant FAQ entries found."

    context = "\n\n---\n\n".join([
        f"FAQ Entry {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(results)
    ])

    return f"Found {len(results)} detailed FAQ entries:\n\n{context}"


@tool
def reformulate_query(original_query: str, focus_aspect: str) -> str:
    """Reformulate the query to focus on a specific aspect.
    Use this when you need to search for a different angle of the question.

    Args:
        original_query: The original user question
        focus_aspect: The specific aspect to focus on (e.g., "pricing", "activation", "troubleshooting")

    Returns:
        A reformulated query focused on the specified aspect
    """
    reformulated = f"{focus_aspect} related to {original_query}"
    results = store.similarity_search(reformulated, k=3)

    if not results:
        return f"No results found for aspect: {focus_aspect}"

    context = "\n\n---\n\n".join([
        f"Entry {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(results)
    ])

    return f"Results for '{focus_aspect}' aspect:\n\n{context}"


# Tools list
tools = [search_faq, search_detailed_faq, reformulate_query]

# Configure Groq model using ChatGroq (LangChain native)
# Available models: llama-3.3-70b-versatile, llama-3.1-70b-versatile, mixtral-8x7b-32768
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

# System prompt for FAQ assistant
system_prompt = """You are a helpful FAQ assistant for Lauki Phones with access to a knowledge base.

Your goal is to answer user questions accurately using the available tools.

Guidelines:
1. Start by using the search_faq tool to find relevant information
2. If the initial search doesn't provide enough info, use search_detailed_faq for more results
3. If the query is complex, use reformulate_query to search different aspects
4. Synthesize information from multiple tool calls if needed
5. Always provide a clear, concise answer based on the retrieved information
6. If you cannot find relevant information, clearly state that

Think step-by-step and use tools strategically to provide the best answer."""

# Create the agent using LangGraph
agent = create_react_agent(
    model=model,
    tools=tools,
    prompt=system_prompt
)


def langgraph_agent_groq(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    result = agent.invoke({"messages": [("human", user_input)]})
    return result['messages'][-1].content


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = langgraph_agent_groq(json.loads(args.payload))
    print(response)

Overwriting langgraph_agents_groq.py


#### Invoking local agent
#### 调用本地智能体

Let's test the agent locally before deploying to AgentCore Runtime.

让我们在部署到 AgentCore Runtime 之前先在本地测试智能体。

In [25]:
# Test locally - Windows uses double quotes for JSON
!python langgraph_agents_groq.py "{\"prompt\": \"Explain roaming activation.\"}"

**Roaming activation on Lauki Phones**

1. **Buy a roaming pack**  
   - Go to the app or website and purchase a region‑specific roaming pack (e.g., daily 500 MB for ₹199).  
   - The pack is tied to your account and will be automatically applied when you travel.

2. **Enable roaming on your device**  
   - Open the device settings → Mobile data → “Roaming” and toggle it on.  
   - Alternatively, you can enable it via the Lauki app’s “Roaming” toggle.

3. **Automatic credential push**  
   - Once roaming is enabled, the system pushes the necessary roaming credentials to the visited network within minutes.  
   - Your device will register on the foreign network and start using the purchased pack.

4. **Verify**  
   - Check the app dashboard for a “Roaming” status indicator.  
   - You’ll see real‑time data usage and any geo‑fenced alerts.

That’s all you need to activate roaming on a Lauki Phone.


d:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\04-strands-with-groq-model\langgraph_agents_groq.py:152: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [26]:
# Test more queries
!python langgraph_agents_groq.py "{\"prompt\": \"What plans does Lauki Phones offer?\"}"

**Lauki Phones plans**

Lauki Phones offers a range of plans to suit different needs:

| Plan type | Key features |
|-----------|--------------|
| **Pre‑paid** | Pay‑as‑you‑go, no contract, flexible data & voice limits. |
| **Post‑paid** | Monthly billing, fixed data/voice allowances, auto‑renewal. |
| **Family** | Shared data pool for multiple lines, often with discounted rates. |
| **Enterprise** | Business‑grade plans with device‑management APIs, priority support, and bulk‑device discounts. |
| **Data‑only** | Pure data plans for tablets, IoT devices, or tethering. |

All plans specify data limits, voice allowances, roaming eligibility, renewal rules, and addon compatibility. Enterprise plans additionally provide APIs for device management and priority support.


d:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\04-strands-with-groq-model\langgraph_agents_groq.py:152: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## Preparing your agent for deployment on AgentCore Runtime
## 准备将智能体部署到 AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

现在让我们将智能体部署到 AgentCore Runtime。为此我们需要：
* 使用 `from bedrock_agentcore.runtime import BedrockAgentCoreApp` 导入 Runtime App
* 在代码中使用 `app = BedrockAgentCoreApp()` 初始化 App
* 使用 `@app.entrypoint` 装饰器装饰调用函数
* 使用 `app.run()` 让 AgentCore Runtime 控制智能体的运行

In [ ]:
%%writefile langgraph_agents_groq.py
"""
FAQ Agent using LangChain + ChatGroq with AgentCore Runtime
This version uses LangGraph for reliable tool calling with Groq models.
"""
import csv
import os
import argparse
import json
from typing import List

from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
from bedrock_agentcore.runtime import BedrockAgentCoreApp

# Initialize AgentCore App
app = BedrockAgentCoreApp()

# Load environment variables
_ = load_dotenv()


def load_faq_csv(path: str) -> List[Document]:
    """Load FAQ data from CSV file"""
    docs = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            q = row["question"].strip()
            a = row["answer"].strip()
            docs.append(Document(page_content=f"Q: {q}\nA: {a}"))
    return docs


# Initialize FAQ knowledge base
script_dir = os.path.dirname(os.path.abspath(__file__))
csv_path = os.path.join(script_dir, "lauki_qna.csv")
docs = load_faq_csv(csv_path)

# Setup embeddings and vector store
emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
chunks = splitter.split_documents(docs)
store = FAISS.from_documents(chunks, emb)


# Define tools for the agent
@tool
def search_faq(query: str) -> str:
    """Search the FAQ knowledge base for relevant information.
    Use this tool when the user asks questions about Lauki Phones products, services, or policies.

    Args:
        query: The search query to find relevant FAQ entries

    Returns:
        Relevant FAQ entries that might answer the question
    """
    results = store.similarity_search(query, k=3)

    if not results:
        return "No relevant FAQ entries found."

    context = "\n\n---\n\n".join([
        f"FAQ Entry {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(results)
    ])

    return f"Found {len(results)} relevant FAQ entries:\n\n{context}"


@tool
def search_detailed_faq(query: str) -> str:
    """Search the FAQ knowledge base with more results for complex queries.
    Use this when the initial search doesn't provide enough information.

    Args:
        query: The search query

    Returns:
        More comprehensive FAQ entries (5 results)
    """
    results = store.similarity_search(query, k=5)

    if not results:
        return "No relevant FAQ entries found."

    context = "\n\n---\n\n".join([
        f"FAQ Entry {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(results)
    ])

    return f"Found {len(results)} detailed FAQ entries:\n\n{context}"


@tool
def reformulate_query(original_query: str, focus_aspect: str) -> str:
    """Reformulate the query to focus on a specific aspect.
    Use this when you need to search for a different angle of the question.

    Args:
        original_query: The original user question
        focus_aspect: The specific aspect to focus on (e.g., "pricing", "activation", "troubleshooting")

    Returns:
        A reformulated query focused on the specified aspect
    """
    reformulated = f"{focus_aspect} related to {original_query}"
    results = store.similarity_search(reformulated, k=3)

    if not results:
        return f"No results found for aspect: {focus_aspect}"

    context = "\n\n---\n\n".join([
        f"Entry {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(results)
    ])

    return f"Results for '{focus_aspect}' aspect:\n\n{context}"


# Tools list
tools = [search_faq, search_detailed_faq, reformulate_query]

# Configure Groq model using ChatGroq (LangChain native)
# Available models: llama-3.3-70b-versatile, llama-3.1-70b-versatile, mixtral-8x7b-32768
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

# System prompt for FAQ assistant
system_prompt = """You are a helpful FAQ assistant for Lauki Phones with access to a knowledge base.

Your goal is to answer user questions accurately using the available tools.

Guidelines:
1. Start by using the search_faq tool to find relevant information
2. If the initial search doesn't provide enough info, use search_detailed_faq for more results
3. If the query is complex, use reformulate_query to search different aspects
4. Synthesize information from multiple tool calls if needed
5. Always provide a clear, concise answer based on the retrieved information
6. If you cannot find relevant information, clearly state that

Think step-by-step and use tools strategically to provide the best answer."""

# Create the agent using LangGraph
agent = create_react_agent(
    model=model,
    tools=tools,
    prompt=system_prompt
)


@app.entrypoint
def langgraph_agent_groq(payload, context=None):
    """
    Invoke the agent with a payload
    Handler for agent invocation in AgentCore runtime
    """
    print("Received payload:", payload)
    print("Context:", context)
    
    user_input = payload.get("prompt")
    result = agent.invoke({"messages": [("human", user_input)]})
    response = result['messages'][-1].content
    
    print("Result:", response)
    return {"result": response}


if __name__ == "__main__":
    app.run()

## What happens behind the scenes?
## 幕后发生了什么？

When you use `BedrockAgentCoreApp`, it automatically:

当您使用 `BedrockAgentCoreApp` 时，它会自动：

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

* 创建一个监听 8080 端口的 HTTP 服务器
* 实现处理智能体请求所需的 `/invocations` 端点
* 实现用于健康检查的 `/ping` 端点
* 处理正确的内容类型和响应格式
* 按照 AWS 标准进行错误处理

## Deploying the agent to AgentCore Runtime
## 将智能体部署到 AgentCore Runtime

**Important:** When deploying to AgentCore Runtime, you need to pass the `GROQ_API_KEY` as an environment variable.

**重要：** 部署到 AgentCore Runtime 时，您需要将 `GROQ_API_KEY` 作为环境变量传递。

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import boto3
import json
import os

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()

agent_name = "langgraph_groq_faq_agent"
response = agentcore_runtime.configure(
    entrypoint="langgraph_agents_groq.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    # Pass GROQ_API_KEY as environment variable
    environment_variables={
        "GROQ_API_KEY": os.getenv("GROQ_API_KEY")
    }
)
response

### Launching agent to AgentCore Runtime
### 将智能体启动到 AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime.

现在我们已经有了 Dockerfile，让我们将智能体启动到 AgentCore Runtime。

In [ ]:
launch_result = agentcore_runtime.launch()

### Checking for the AgentCore Runtime Status
### 检查 AgentCore Runtime 状态

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invoking AgentCore Runtime
### 调用 AgentCore Runtime

Now let's test our FAQ agent with some questions about Lauki Phones.

现在让我们用一些关于 Lauki Phones 的问题来测试我们的 FAQ 智能体。

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "How do I activate international roaming?"})
invoke_response

In [ ]:
# Test more FAQ questions
questions = [
    "What plans does Lauki Phones offer?",
    "How do I check my data balance?",
    "Does Lauki Phones support 5G?",
    "How long does porting take?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Question: {q}")
    print(f"{'='*60}")
    response = agentcore_runtime.invoke({"prompt": q})
    print(f"Answer: {response}")

### Processing invocation results
### 处理调用结果

In [ ]:
from IPython.display import Markdown, display
import json

response_text = invoke_response['response'][0]
display(Markdown(response_text))

### Invoking AgentCore Runtime with boto3
### 使用 boto3 调用 AgentCore Runtime

In [ ]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What happens if my bill is overdue?"})
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## Cleanup (Optional)
## 清理（可选）

Let's now clean up the AgentCore Runtime created.

现在让我们清理已创建的 AgentCore Runtime。

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

# Congratulations!
# 恭喜！

You have successfully deployed a RAG-based FAQ agent using LangGraph with Groq models on Amazon Bedrock AgentCore Runtime!

您已成功在 Amazon Bedrock AgentCore Runtime 上部署了一个使用 LangGraph 和 Groq 模型的基于 RAG 的 FAQ 智能体！

## Key Takeaways
## 关键要点

- **Groq** provides ultra-fast LLM inference with models like Llama 3.3 70B
- **ChatGroq** is LangChain's native integration for Groq with reliable tool calling
- **LangGraph** provides a robust ReAct agent implementation
- **FAISS + HuggingFace Embeddings** enable efficient vector similarity search for RAG
- **AgentCore Runtime** simplifies deployment and scaling of AI agents

- **Groq** 通过 Llama 3.3 70B 等模型提供超快的 LLM 推理
- **ChatGroq** 是 LangChain 对 Groq 的原生集成，提供可靠的工具调用
- **LangGraph** 提供健壮的 ReAct 智能体实现
- **FAISS + HuggingFace Embeddings** 为 RAG 提供高效的向量相似度搜索
- **AgentCore Runtime** 简化了 AI 智能体的部署和扩展